## **Task 2**

In [ ]:
!pip install --no-cache-dir git+https://github.com/AmenRa/ranx.git

  Cloning https://github.com/AmenRa/ranx.git to /tmp/pip-req-build-859hfsos
  Running command git clone --filter=blob:none --quiet https://github.com/AmenRa/ranx.git /tmp/pip-req-build-859hfsos
  Resolved https://github.com/AmenRa/ranx.git to commit f60b1ac230ab97bd3c6ad2c210f79882293b306d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 268.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 313.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 296.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 293.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 143.6 MB/s eta 0:00:00
  Created whe

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from ranx import Run, Qrels

path_miniLM = "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2025-05-22_21-48-32/ranking.run"
path_tinyBERT = "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-23_08-10-50/ranking.run"
path_roberta = "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/cross-encoder-distilroberta-base-2025-05-23_09-18-00/ranking.run"

qrels_path = "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/qrels/2019qrels-pass.txt"

In [ ]:
run1 = Run.from_file(path_miniLM, kind="trec", name="MiniLM")
run2 = Run.from_file(path_tinyBERT, kind="trec", name="TinyBERT")
# run3 = Run.from_file(path_roberta, kind="trec", name="DistilRoBERTa")

qrels = Qrels.from_file(qrels_path, kind="trec")

In [ ]:
from ranx import Run

run3 = Run.from_file(
    "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base-2025-05-23_09-18-00/ranking.run",
    kind="trec",
    name="DistilRoBERTa"
)

In [ ]:
pip install --upgrade ranx

In [ ]:
from ranx import fuse

runs = [run1, run2, run3]

fused_methods = {
    "Reciprocal Rank Fusion (k=60)": fuse(runs, method="rrf"),              # default k=60
    "Reciprocal Rank Fusion (k=30)": fuse(runs, method="rrf", params={"k": 30}),
    "Minimum": fuse(runs, method="min"),
    "Maximum": fuse(runs, method="max"),
    "MNZ": fuse(runs, method="mnz"),
}

In [ ]:
from ranx import evaluate

for name, fused_run in fused_methods.items():
    print(f"{name}")
    scores = evaluate(qrels=qrels, run=fused_run, metrics=["ndcg@10", "recall@100", "map@1000"])
    print(scores)
    print("-" * 50)

Reciprocal Rank Fusion (k=60)
{'ndcg@10': np.float64(0.7002495906751198), 'recall@100': np.float64(0.5132333533967706), 'map@1000': np.float64(0.4555198337072335)}
--------------------------------------------------
Reciprocal Rank Fusion (k=30)
{'ndcg@10': np.float64(0.7021726791105982), 'recall@100': np.float64(0.5121726144983443), 'map@1000': np.float64(0.4541655188049011)}
--------------------------------------------------
Minimum
{'ndcg@10': np.float64(0.6833114387384082), 'recall@100': np.float64(0.4858480990281992), 'map@1000': np.float64(0.42867563427305333)}
--------------------------------------------------
Maximum
{'ndcg@10': np.float64(0.6446675104831884), 'recall@100': np.float64(0.49995218249882406), 'map@1000': np.float64(0.4214219749015709)}
--------------------------------------------------
MNZ
{'ndcg@10': np.float64(0.6921199048418143), 'recall@100': np.float64(0.5119369068198271), 'map@1000': np.float64(0.45153750074877375)}
-------------------------------------------

In [ ]:
best_fused_run = fused_methods["Reciprocal Rank Fusion (k=30)"]
best_fused_run.save(
    "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/task2-fusion.run",
    kind="trec"
)

## **Task 3**

In [ ]:
base_path = "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/"

In [ ]:
!mkdir -p trec2019-data
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz -P trec2019-data
!gunzip trec2019-data/msmarco-test2019-queries.tsv.gz

--2025-05-25 08:55:05--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4276 (4.2K) [application/x-gzip]
Saving to: ‘trec2019-data/msmarco-test2019-queries.tsv.gz’

msmarco-test2019-qu 100%[===================>]   4.18K  --.-KB/s    in 0s      

2025-05-25 08:55:05 (740 MB/s) - ‘trec2019-data/msmarco-test2019-queries.tsv.gz’ saved [4276/4276]



In [ ]:
!mkdir -p trec2019-data
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz -P trec2019-data
!gunzip trec2019-data/msmarco-passagetest2019-top1000.tsv.gz

--2025-05-25 08:55:08--  https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634062 (25M) [application/x-gzip]
Saving to: ‘trec2019-data/msmarco-passagetest2019-top1000.tsv.gz’

msmarco-passagetest 100%[===================>]  25.40M  43.9MB/s    in 0.6s    

2025-05-25 08:55:08 (43.9 MB/s) - ‘trec2019-data/msmarco-passagetest2019-top1000.tsv.gz’ saved [26634062/26634062]



In [ ]:
import json
import gzip
from collections import defaultdict

queries = {}
with open("trec2019-data/msmarco-test2019-queries.tsv", "r", encoding="utf8") as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query

passage_cand = {}
with open("trec2019-data/msmarco-passagetest2019-top1000.tsv", "r", encoding="utf8") as fIn:
    for line in fIn:
        qid, pid, _, passage = line.strip().split("\t")
        passage_cand.setdefault(qid, []).append([pid, passage])

qrels_path = base_path + "qrels/2019qrels-pass.txt"
relevant_docs = defaultdict(lambda: defaultdict(int))
with open(qrels_path, "r") as fIn:
    for line in fIn:
        qid, _, pid, score = line.strip().split()
        if int(score) > 0:
            relevant_docs[qid][pid] = int(score)

relevant_qid = [qid for qid in queries if len(relevant_docs[qid]) > 0]

In [ ]:
synonym_map = {
    "renewable": ["sustainable", "green"],
    "education": ["learning", "schooling"],
    "climate": ["weather", "environment"],
    "tax": ["levy", "subsidy"],
    "policy": ["regulation", "law"],
    "health": ["wellness", "medical"],
    "technology": ["innovation", "science"],
    "internet": ["web", "online"]
}

def expand_query(query):
    tokens = query.split()
    expansion = []
    for word in tokens:
        if word.lower() in synonym_map:
            expansion.extend(synonym_map[word.lower()])
    return query + " " + " ".join(expansion)

expanded_queries = {qid: expand_query(queries[qid]) for qid in queries}

In [ ]:
from sentence_transformers import CrossEncoder

model_path = base_path + "finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2025-05-22_21-48-32"
model = CrossEncoder(model_path, max_length=512)

In [ ]:
import tqdm, operator

run_expanded = {}

for qid in tqdm.tqdm(relevant_qid):
    query = expanded_queries[qid]
    candidates = passage_cand[qid]
    pids = [pid for pid, _ in candidates]
    texts = [text for _, text in candidates]

    inputs = [[query, passage] for passage in texts]
    scores = model.predict(inputs).tolist()

    run_expanded[qid] = sorted(zip(pids, scores), key=operator.itemgetter(1), reverse=True)

100%|██████████| 43/43 [17:33<00:00, 24.50s/it]


In [ ]:
import tqdm, operator

run_expanded = {}

for qid in tqdm.tqdm(relevant_qid):
    query = expanded_queries[qid]
    candidates = passage_cand[qid]
    pids = [pid for pid, _ in candidates]
    texts = [text for _, text in candidates]

    inputs = [[query, passage] for passage in texts]
    scores = model.predict(inputs).tolist()

    run_expanded[qid] = sorted(zip(pids, scores), key=operator.itemgetter(1), reverse=True)

100%|██████████| 43/43 [17:28<00:00, 24.39s/it]


In [ ]:
run_file_path = base_path + "task3-query-expanded.run"

with open(run_file_path, "w") as f:
    for qid in run_expanded:
        for rank, (pid, score) in enumerate(run_expanded[qid]):
            f.write(f"{qid} Q0 {pid} {rank} {score} task3_expanded\n")

In [ ]:
from ranx import Run, Qrels, evaluate

run = Run.from_file(run_file_path, kind="trec")
qrels = Qrels.from_file(qrels_path, kind="trec")

scores = evaluate(qrels=qrels, run=run, metrics=["ndcg@10", "recall@100", "map@1000"])
print("Task 3 – Results with Expanded Queries:")
print(scores)

Task 3 – Results with Expanded Queries:
{'ndcg@10': np.float64(0.661433770543588), 'recall@100': np.float64(0.4954028069562599), 'map@1000': np.float64(0.42576423099557764)}


In [ ]:
run1 = Run.from_file(path_miniLM, kind="trec", name="MiniLM")
run2 = Run.from_file(path_tinyBERT, kind="trec", name="TinyBERT")
from ranx import Run

run3 = Run.from_file(
    "/content/gdrive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base-2025-05-23_09-18-00/ranking.run",
    kind="trec",
    name="DistilRoBERTa"
)

run_minilm = run1
run_tinybert = run2
run_distilroberta = run3

In [ ]:
from ranx import fuse, evaluate

pairwise_results = {}

rrf_params = {"k": 30}

# MiniLM + TinyBERT
fused_1 = fuse([run_minilm, run_tinybert], method="rrf", params=rrf_params)
pairwise_results["MiniLM + TinyBERT"] = evaluate(qrels=qrels, run=fused_1, metrics=["ndcg@10", "recall@100", "map@1000"])

# MiniLM + DistilRoBERTa
fused_2 = fuse([run_minilm, run_distilroberta], method="rrf", params=rrf_params)
pairwise_results["MiniLM + DistilRoBERTa"] = evaluate(qrels=qrels, run=fused_2, metrics=["ndcg@10", "recall@100", "map@1000"])

# TinyBERT + DistilRoBERTa
fused_3 = fuse([run_tinybert, run_distilroberta], method="rrf", params=rrf_params)
pairwise_results["TinyBERT + DistilRoBERTa"] = evaluate(qrels=qrels, run=fused_3, metrics=["ndcg@10", "recall@100", "map@1000"])

for pair, scores in pairwise_results.items():
    print(f"{pair}:")
    print(scores)
    print("-" * 50)

MiniLM + TinyBERT:
{'ndcg@10': np.float64(0.7033019088129966), 'recall@100': np.float64(0.5088437923182634), 'map@1000': np.float64(0.45482859872978965)}
--------------------------------------------------
MiniLM + DistilRoBERTa:
{'ndcg@10': np.float64(0.6758188631245787), 'recall@100': np.float64(0.5035138750093001), 'map@1000': np.float64(0.4357143810819189)}
--------------------------------------------------
TinyBERT + DistilRoBERTa:
{'ndcg@10': np.float64(0.6940366077237328), 'recall@100': np.float64(0.5056593010098537), 'map@1000': np.float64(0.4504648531934809)}
--------------------------------------------------


## **Task 4**

In [ ]:
!pip install --quiet ranx transformers sentence-transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!mkdir -p data
!wget -q https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz -P data
!wget -q https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz -P data
!wget -q https://trec.nist.gov/data/deep/2019qrels-pass.txt -O data/2019qrels-pass.txt
!gunzip -f data/*.gz

In [ ]:
import pandas as pd

queries = pd.read_csv(
    "data/msmarco-test2019-queries.tsv",
    sep="\t",
    header=None,
    names=["qid", "query"]
)
cands = pd.read_csv(
    "data/msmarco-passagetest2019-top1000.tsv",
    sep="\t",
    header=None,
    names=["qid", "pid", "text"]
)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install huggingface_hub
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineG

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "tiiuae/falcon-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

prompt = 'Query: "example query"\nFinal expanded query:'
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=16)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Query: "example query"
Final expanded query: "example query"
The query is expanded to include all of the search


In [ ]:
prompt_template = """
You are an IR assistant. Reformulate the user query using chain-of-thought:
Query: "{query}"
Step 1: Identify key terms.
Step 2: [Your reasoning here…]
Final expanded query:"""

def expand_query(q):
    inp = prompt_template.format(query=q)
    tokens = tokenizer(inp, return_tensors="pt").to(model.device)
    out = model.generate(**tokens, max_new_tokens=64, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("Final expanded query:")[-1].strip()

In [ ]:
import pandas as pd

queries = pd.read_csv("data/msmarco-test2019-queries.tsv", sep="\t", header=None, names=["qid","query"])
expansions = {}
for _, row in queries.iterrows():
    expansions[row.qid] = expand_query(row.query)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


In [ ]:
from sentence_transformers import CrossEncoder

ce_path = "/content/gdrive/MyDrive/cross-encoder-ms-marco-MiniLM-L-2-v2-2025-05-22_21-48-32"
ce = CrossEncoder(ce_path)

In [ ]:
cands = pd.read_csv("data/msmarco-passagetest2019-top1000.tsv",
                    sep="\t", header=None, names=["qid","pid","text"])
import os

os.makedirs("expansion_runs", exist_ok=True)
with open("expansion_runs/expansion.run", "w") as runfile:
    for qid, expanded in expansions.items():
        subset = cands[cands.qid == qid]
        pairs = [(expanded, txt) for txt in subset.text.tolist()]
        scores = ce.predict(pairs, show_progress_bar=False)
        subset = subset.assign(score=scores).sort_values("score", ascending=False)
        for rank, row in enumerate(subset.itertuples(), start=1):
            runfile.write(f"{qid} Q0 {row.pid} {rank} {row.score:.4f} expansion\n")

In [ ]:
from ranx import Run, Qrels, evaluate

qrels = Qrels.from_file("data/2019qrels-pass.txt", fmt="trec")
run = Run.from_file("expansion_runs/expansion.run", kind="trec", name="LLM-CoT-Expansion")
metrics = evaluate(qrels=qrels, run=run, metrics=["ndcg@10", "recall@100", "map@1000"])
print(metrics)